# DGS Genomic Language Model Adapters

This notebook introduces how DGS works with genomic language models (gLMs). The goal is not to train a large model from scratch, but to understand how DGS wraps external genome foundation models behind a unified sequence-encoder interface.


## Tutorial series map

This notebook is part of the `Tutorials/0_DGS_usage_models` sequence:
- `0_DGS_data.ipynb`: data, IO, intervals, targets, and loaders
- `1_DGS_models.ipynb`: binary classification and regression scalar models
- `2_DGS_profile_models.ipynb`: profile/count and long-sequence track models
- **`3_DGS_gLMs.ipynb`: genomic language model adapters and downstream heads**
- `4_DGS_explain.ipynb`: interpretation methods, attribution artifacts, and motif discovery

The notebooks are designed to be read in order, but each one keeps its examples self-contained and guarded against missing optional dependencies.


## Audience, prerequisites, and learning goals

This notebook is for users who have already seen DGS data loaders and scalar/profile model workflows, and now want to connect external genome foundation models to DGS-style downstream tasks.

Prerequisites:

- Basic DGS sequence-input conventions.
- Basic PyTorch module concepts.
- Familiarity with tokenizers and embeddings is helpful but not required.

By the end, you should be able to:

- Explain the role and boundary of DGS gLM adapters.
- Inspect supported gLM presets and choose an appropriate pooling or output mode.
- Use a lightweight mock encoder to understand `FoundationSequenceAdapter` inputs, outputs, and encoder freezing.
- Optionally load DNABERT, DNABERT-2, Nucleotide Transformer, GPN, and Evo2 adapters.
- Keep large-model downloads, memory requirements, and downstream task-head training as separate steps.


## Outline

1. Explain what gLM adapters solve in DGS.
2. Set up a safe tutorial environment.
3. Inspect the gLM preset registry.
4. Learn the encoder, tokenizer, and pooling contract.
5. Understand DNABERT-style k-mer preprocessing.
6. Optionally build real adapters with the factory function.
7. Connect embeddings to a downstream task head.
8. Sketch a lightweight variant-effect pattern.
9. Complete an exercise and review common pitfalls.


## Series-level conventions

Across these notebooks:

- Core examples use tiny synthetic data or local fixtures and avoid model downloads.
- Optional real-data or external-checkpoint cells are disabled with `RUN_*` flags until you edit paths and opt in.
- Loader sequence batches are usually `(batch, length, 4)`; CNN-style models usually expect `(batch, 4, length)` after `transpose(1, 2)`.
- Scalar targets use `(batch, tasks)`; profile targets use `(batch, tracks, profile_length)`.
- Environment guards print skip messages when optional packages such as `torch`, `pysam`, `pyBigWig`, `transformers`, or `evo2` are unavailable.


## 1. What problem do gLMs solve in DGS?

Traditional DGS models usually consume one-hot DNA sequences directly, with shapes such as `(batch, 4, sequence_length)` or `(batch, sequence_length, 4)`. gLMs first encode DNA sequences into token-level or sequence-level embeddings, which can then be used for:

- Zero-shot or few-shot sequence representations.
- Input features for downstream classification, regression, or profile models.
- Reference-versus-alternate embedding or logits comparisons for variant analysis.
- Pretrained feature initialization in motif, cell-type, MPRA, and related tasks.

DGS provides the adapter layer and factory functions only. The actual checkpoints, model weights, and some runtime dependencies still come from external projects. This keeps the base DGS installation lightweight.

## 2. Environment setup

The core tutorial cells do not download any large model by default. Real Hugging Face / Evo2 model loading is placed in optional cells later in the notebook, and those cells only run after you explicitly set an environment variable.

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import sys

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "DGS").is_dir() and (candidate / "Tutorials").is_dir():
            return candidate
    raise RuntimeError("Cannot locate the DeepGeSeq repository root.")

def importable_package(name):
    if importlib.util.find_spec(name) is None:
        return False, "not installed"
    try:
        importlib.import_module(name)
        return True, None
    except Exception as exc:
        # A failed import can leave partial modules behind in long-lived kernels.
        for module_name in list(sys.modules):
            if module_name == name or module_name.startswith(name + "."):
                sys.modules.pop(module_name, None)
        return False, f"{type(exc).__name__}: {exc}"

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

TORCH_AVAILABLE, TORCH_IMPORT_ERROR = importable_package("torch")
if TORCH_AVAILABLE:
    TORCH_AVAILABLE, TORCH_NN_IMPORT_ERROR = importable_package("torch.nn")
    TORCH_IMPORT_ERROR = TORCH_NN_IMPORT_ERROR
TRANSFORMERS_AVAILABLE = importlib.util.find_spec("transformers") is not None
EVO2_AVAILABLE = importlib.util.find_spec("evo2") is not None
RUN_GLM_DOWNLOADS = os.environ.get("DGS_RUN_GLM_DOWNLOADS") == "1"

print(f"Repository: {REPO_ROOT}")
print(f"torch available: {TORCH_AVAILABLE}")
if TORCH_IMPORT_ERROR:
    print(f"torch import issue: {TORCH_IMPORT_ERROR}")
print(f"transformers available: {TRANSFORMERS_AVAILABLE}")
print(f"evo2 available: {EVO2_AVAILABLE}")
print(f"run optional model downloads: {RUN_GLM_DOWNLOADS}")


If `torch available` is `False`, the current kernel is not a usable DGS runtime. Install the base dependencies first; real gLM adapters also require the optional `llm` dependencies.

```bash
pip install -e .
pip install -e ".[llm]"
```

Evo2 has separate CUDA and package-version requirements, so it is best installed following the upstream Evo2 instructions.

## 3. Inspect the DGS gLM presets

The unified entry point is `build_genomic_language_model_adapter(model_family, **kwargs)`. The `model_family` argument can be an alias such as `dnabert2`, `nt-v2-50m`, `gpn`, or `evo2-7b`.

In [ ]:
if TORCH_AVAILABLE:
    from DGS.Model import GLM_MODEL_PRESETS

    rows = []
    for name, preset in sorted(GLM_MODEL_PRESETS.items()):
        rows.append((
            name,
            preset.get("family"),
            preset.get("provider"),
            preset.get("model_name_or_path", preset.get("model_name", "")),
            preset.get("pooling"),
        ))
else:
    rows = [
        ("dnabert-3mer", "DNABERT", "transformers", "zhihan1996/DNA_bert_3", "mean"),
        ("dnabert2", "DNABERT2", "transformers", "zhihan1996/DNABERT-2-117M", "mean"),
        ("nt-v2-50m", "NucleotideTransformer", "transformers", "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species", "mean"),
        ("gpn", "GPN", "transformers", "songlab/gpn-brassicales", "none"),
        ("evo2-7b", "Evo2", "evo2", "evo2_7b", "none"),
    ]

print(f"Known presets: {len(rows)}")
print(f"{'preset':<24} {'family':<24} {'provider':<14} {'pooling':<8} model")
print("-" * 110)
for preset, family, provider, model, pooling in rows[:40]:
    print(f"{preset:<24} {family:<24} {provider:<14} {pooling:<8} {model}")

A practical way to read the preset list is by task and runtime expectation:

- `dnabert-3mer` / `dnabert-4mer` / `dnabert-5mer` / `dnabert-6mer`: original DNABERT-style checkpoints. DGS converts raw DNA into overlapping k-mer tokens before tokenization.
- `dnabert2`: DNABERT-2, often used as a medium-sized general DNA encoder.
- `nt-v2-50m`, `nt-v2-100m`, `nt-v2-250m`, `nt-v2-500m`: Nucleotide Transformer family presets.
- `gpn` / `phylogpn` / `gpn-star`: GPN-family models, often used for masked-language-model logits or species/phylogeny-aware modeling.
- `evo2-*`: Evo2 presets. These can return logits or embeddings from selected intermediate layers, but the runtime and memory requirements are usually higher.

## 4. Core adapter concepts: encoder, tokenizer, pooling

`FoundationSequenceAdapter` performs three steps:

1. If the input is a DNA string, call the tokenizer and produce tensors.
2. Call the external encoder and extract `last_hidden_state`, `logits`, or another requested tensor.
3. Convert token-level representations into the requested output using the `pooling` argument.

Common pooling modes:

- `mean`: average across sequence positions, returning `(batch, hidden_dim)`.
- `cls`: use the first token, returning `(batch, hidden_dim)`.
- `max`: take the maximum across sequence positions, returning `(batch, hidden_dim)`.
- `none`: do not aggregate; keep shapes such as `(batch, token_length, hidden_dim)` or logits.

In [ ]:
if TORCH_AVAILABLE:
    import torch
    from DGS.Model import FoundationSequenceAdapter

    class TinyTokenizer:
        """Convert A/C/G/T/N into integer tokens, mimicking a Hugging Face tokenizer."""
        vocab = {"A": 0, "C": 1, "G": 2, "T": 3, "N": 4}

        def __call__(self, sequences, return_tensors="pt", padding=True):
            max_len = max(len(seq) for seq in sequences)
            token_rows = []
            mask_rows = []
            for seq in sequences:
                ids = [self.vocab.get(base, 4) for base in seq.upper()]
                pad = max_len - len(ids)
                token_rows.append(ids + [4] * pad)
                mask_rows.append([1] * len(ids) + [0] * pad)
            return {
                "input_ids": torch.tensor(token_rows, dtype=torch.long),
                "attention_mask": torch.tensor(mask_rows, dtype=torch.long),
            }

    class TinyGenomeEncoder(torch.nn.Module):
        """Tiny encoder that uses an embedding layer to mimic gLM token representations."""
        def __init__(self, vocab_size=5, hidden_dim=6):
            super().__init__()
            self.embedding = torch.nn.Embedding(vocab_size, hidden_dim)

        def forward(self, input_ids, attention_mask=None):
            hidden = self.embedding(input_ids)
            return {"last_hidden_state": hidden}

    sequences = ["ACGTACGT", "TGCANNNN"]
    adapter = FoundationSequenceAdapter(
        encoder=TinyGenomeEncoder(),
        tokenizer=TinyTokenizer(),
        pooling="mean",
        freeze_encoder=True,
    )
    embeddings = adapter(sequences)

    print("Input sequences:", sequences)
    print("Mean-pooled embedding shape:", tuple(embeddings.shape))
    print("Encoder frozen:", all(not p.requires_grad for p in adapter.encoder.parameters()))
else:
    print("Skip: this demonstration requires torch.")

In [ ]:
if TORCH_AVAILABLE:
    for pooling in ["mean", "cls", "max", "none"]:
        adapter = FoundationSequenceAdapter(
            encoder=TinyGenomeEncoder(),
            tokenizer=TinyTokenizer(),
            pooling=pooling,
        )
        y = adapter(["ACGT", "TGCA"])
        print(f"pooling={pooling:<4} -> shape={tuple(y.shape)}")
else:
    print("Skip: this demonstration requires torch.")

For real downstream tasks, `mean` and `cls` are common when you want a single vector for the whole sequence. `none` is useful when you need token-level information, or when you plan to attach a CNN, Transformer, or attention head later. Pooling is a modeling assumption, not just an implementation detail: should the full sequence be summarized by one vector, or do you need to preserve position-level information?

## 5. DNABERT v1 k-mer preprocessing

Original DNABERT v1 checkpoints usually do not consume `ACGTAC` directly. They consume overlapping k-mer strings. For example, 3-mer preprocessing converts `ACGTAC` into:

```text
ACG CGT GTA TAC
```

DGS performs this step automatically in `DNABERTAdapter` when `pretokenized=False`.

In [ ]:
def sequence_to_kmer_string(sequence, kmer):
    sequence = str(sequence).strip().upper()
    if len(sequence) < kmer:
        raise ValueError("sequence length must be >= kmer")
    return " ".join(sequence[i:i + kmer] for i in range(len(sequence) - kmer + 1))

for k in [3, 4, 5, 6]:
    print(f"{k}-mer:", sequence_to_kmer_string("ACGTACGT", k))

In [ ]:
if TORCH_AVAILABLE:
    from DGS.Model import DNABERTAdapter

    tokenized_inputs = []

    class RecordingTokenizer(TinyTokenizer):
        def __call__(self, sequences, return_tensors="pt", padding=True):
            tokenized_inputs.extend(sequences)
            # DNABERT tokenizers receive whitespace-separated k-mer strings; here we build fake ids from token counts only.
            max_len = max(len(seq.split()) for seq in sequences)
            return {"input_ids": torch.zeros((len(sequences), max_len), dtype=torch.long)}

    dnabert_like = DNABERTAdapter(
        encoder=TinyGenomeEncoder(),
        tokenizer=RecordingTokenizer(),
        kmer=4,
        pooling="none",
    )
    output = dnabert_like(["ACGTAC"])

    print("Tokenizer received:", tokenized_inputs)
    print("Output shape:", tuple(output.shape))
else:
    print("Skip: this demonstration requires torch.")

If you have already generated k-mer strings elsewhere, pass `pretokenized=True` to the DNABERT builder to avoid applying the conversion twice. DNABERT-2 and Nucleotide Transformer presets do not use this DNABERT v1 k-mer preprocessing logic.

## 6. Build real gLM adapters with the factory function

The following cells are templates for loading external models. They do not run by default because they may download checkpoints, require extra dependencies such as `transformers`, `gpn`, or `evo2`, and may require a GPU.

To run these cells, set the following environment variable before starting the notebook:

```bash
export DGS_RUN_GLM_DOWNLOADS=1
```

In [ ]:
if RUN_GLM_DOWNLOADS:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Install torch before loading DGS gLM adapters.")
    if not TRANSFORMERS_AVAILABLE:
        raise RuntimeError("Install transformers or run `pip install -e .[llm]` first.")

    from DGS.Model import build_genomic_language_model_adapter

    # DNABERT-2: a medium-sized general DNA encoder.
    adapter = build_genomic_language_model_adapter(
        "dnabert2",
        pooling="mean",
        freeze_encoder=True,
    )
    adapter.eval()

    with torch.no_grad():
        embeddings = adapter(["ACGTACGTACGT", "TGCATGCATGCA"])

    print(type(adapter).__name__)
    print(tuple(embeddings.shape))
else:
    print("Skip real DNABERT-2 loading. Set DGS_RUN_GLM_DOWNLOADS=1 to run this cell.")

In [ ]:
if RUN_GLM_DOWNLOADS:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Install torch before loading DGS gLM adapters.")
    if not TRANSFORMERS_AVAILABLE:
        raise RuntimeError("Install transformers or run `pip install -e .[llm]` first.")

    from DGS.Model import build_genomic_language_model_adapter

    # Nucleotide Transformer: switch among 50M/100M/250M/500M models through presets.
    nt_adapter = build_genomic_language_model_adapter(
        "nt-v2-50m",
        pooling="cls",
        freeze_encoder=True,
    )
    nt_adapter.eval()

    with torch.no_grad():
        nt_embeddings = nt_adapter(["ACGTACGTACGT"])

    print(type(nt_adapter).__name__)
    print(tuple(nt_embeddings.shape))
else:
    print("Skip real Nucleotide Transformer loading. Set DGS_RUN_GLM_DOWNLOADS=1 to run this cell.")

In [ ]:
if RUN_GLM_DOWNLOADS:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Install torch before loading DGS gLM adapters.")
    if not TRANSFORMERS_AVAILABLE:
        raise RuntimeError("Install transformers or run `pip install -e .[llm]` first.")

    from DGS.Model import build_genomic_language_model_adapter

    # GPN returns masked-LM logits by default; pooling="none" is common.
    # Some GPN models also require upstream packages, such as gpn.model.
    gpn_adapter = build_genomic_language_model_adapter("gpn", pooling="none")
    gpn_adapter.eval()

    with torch.no_grad():
        logits = gpn_adapter(["ACGTACGTACGT"])

    print(type(gpn_adapter).__name__)
    print(tuple(logits.shape))
else:
    print("Skip real GPN loading. Set DGS_RUN_GLM_DOWNLOADS=1 to run this cell.")

In [ ]:
if RUN_GLM_DOWNLOADS:
    if not TORCH_AVAILABLE:
        raise RuntimeError("Install torch before loading Evo2 adapters.")
    if not EVO2_AVAILABLE:
        raise RuntimeError("Install Evo2 following the upstream instructions first.")

    from DGS.Model import build_genomic_language_model_adapter

    # Evo2 logits mode. Large models are memory-intensive, so start with the smallest usable setup.
    evo2_adapter = build_genomic_language_model_adapter("evo2-1b-base", pooling="none")
    logits = evo2_adapter(["ACGTACGT"])

    print(type(evo2_adapter).__name__)
    print(tuple(logits.shape))
else:
    print("Skip real Evo2 loading. Set DGS_RUN_GLM_DOWNLOADS=1 to run this cell.")

## 7. Connect gLM embeddings to a downstream task head

A common first workflow is to freeze the gLM, encode sequences into embeddings, and train a small task head on top. This quickly tests whether the pretrained representation is useful for your labels. The cell below demonstrates the shape flow with the mock adapter.

In [ ]:
if TORCH_AVAILABLE:
    import torch.nn as nn

    torch.manual_seed(7)

    encoder = FoundationSequenceAdapter(
        encoder=TinyGenomeEncoder(hidden_dim=8),
        tokenizer=TinyTokenizer(),
        pooling="mean",
        freeze_encoder=True,
    )
    head = nn.Sequential(
        nn.LayerNorm(8),
        nn.Linear(8, 2),
    )

    batch_sequences = ["ACGTACGT", "TGCATGCA", "AAAAACCC"]
    with torch.no_grad():
        z = encoder(batch_sequences)
    logits = head(z)

    print("embedding:", tuple(z.shape))
    print("task logits:", tuple(logits.shape))
else:
    print("Skip: this demonstration requires torch.")

Training recommendations:

- First freeze the gLM and train only a small task head, making sure labels, splits, metrics, and batch pipelines are correct.
- Then consider unfreezing the last few layers or using LoRA / adapter tuning; avoid full fine-tuning at the first attempt.
- For genomic tasks, prefer chromosome-based splits to reduce leakage from nearby regions.
- For variant-effect prediction, encode reference and alternate sequences separately, then compare logits or embedding differences.

## 8. A lightweight variant-effect analysis pattern

For a single-nucleotide variant, you can build centered reference and alternate sequences, pass both through a gLM adapter, and compare the outputs. The example below still uses the mock adapter so that the workflow is easy to run.

In [ ]:
def mutate_sequence(sequence, position, alt_base):
    sequence = sequence.upper()
    return sequence[:position] + alt_base.upper() + sequence[position + 1:]

ref_seq = "ACGTACGTACGT"
alt_seq = mutate_sequence(ref_seq, position=5, alt_base="T")
print("ref:", ref_seq)
print("alt:", alt_seq)

if TORCH_AVAILABLE:
    encoder = FoundationSequenceAdapter(
        encoder=TinyGenomeEncoder(hidden_dim=8),
        tokenizer=TinyTokenizer(),
        pooling="mean",
        freeze_encoder=True,
    )
    with torch.no_grad():
        ref_emb, alt_emb = encoder([ref_seq, alt_seq])
    delta = alt_emb - ref_emb
    print("delta embedding shape:", tuple(delta.shape))
    print("delta L2 norm:", float(torch.linalg.vector_norm(delta)))
else:
    print("Skip embedding comparison: this demonstration requires torch.")

In real projects, variant analysis should not stop at a single embedding distance. A stronger pattern is to connect gLM embeddings to a validated task head, then compare reference and alternate predictions for the relevant cell type, peak, MPRA readout, or profile track.

## 9. Exercise

Change the mock adapter above from `pooling="mean"` to `pooling="none"`, then write a small `nn.Conv1d` or attention-pooling module that aggregates token-level embeddings into `(batch, hidden_dim)` before classification.

Question: when a sequence is long and the functional site is short, can `mean` pooling dilute the signal?

In [ ]:
if TORCH_AVAILABLE:
    import torch.nn as nn
    class ConvPoolingHead(nn.Module):
        """Answer scaffold: convert token-level embeddings into sequence-level logits."""
        def __init__(self, hidden_dim, output_dim):
            super().__init__()
            self.conv = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1)
            self.activation = nn.GELU()
            self.classifier = nn.Linear(hidden_dim, output_dim)

        def forward(self, token_embeddings):
            # token_embeddings: (batch, token_length, hidden_dim)
            x = token_embeddings.transpose(1, 2)
            x = self.activation(self.conv(x))
            x = x.amax(dim=-1)
            return self.classifier(x)

    token_encoder = FoundationSequenceAdapter(
        encoder=TinyGenomeEncoder(hidden_dim=8),
        tokenizer=TinyTokenizer(),
        pooling="none",
        freeze_encoder=True,
    )
    head = ConvPoolingHead(hidden_dim=8, output_dim=2)

    with torch.no_grad():
        token_embeddings = token_encoder(["ACGTACGT", "TTTTCCCC"])
    logits = head(token_embeddings)

    print("token embeddings:", tuple(token_embeddings.shape))
    print("logits:", tuple(logits.shape))
else:
    print("Skip: this exercise answer requires torch.")

## Common pitfalls

- `ModuleNotFoundError: transformers`: the base DGS installation does not include gLM runtimes; install `pip install -e ".[llm]"`.
- `trust_remote_code` warnings or prompts: some Hugging Face models require custom code. Enable it only after confirming the model source is trusted.
- GPN loading failures: some presets require upstream Python packages, such as `gpn.model`.
- Evo2 loading failures: Evo2 does not use Hugging Face `AutoModel`; install `evo2` and its CUDA runtime requirements separately.
- Out-of-memory errors: freeze the encoder, reduce batch size, shorten the sequence, choose a smaller preset, or cache embeddings offline first.
- Unexpected output shapes: check `pooling`, `output_key`, and whether the model returns `last_hidden_state` or `logits`.

## Summary

DGS gLM support is best understood as a stable interface: it turns external genomic language models into DGS-callable encoders. A recommended workflow is:

1. Use `GLM_MODEL_PRESETS` to confirm the model source, provider, and default pooling.
2. Use `FoundationSequenceAdapter` or a mock encoder to validate input and output shapes.
3. Freeze the real gLM, extract embeddings, and train a lightweight downstream head.
4. Consider gLM fine-tuning only when the dataset size, memory budget, and validation results justify it.

This separates the question “is the pretrained representation useful?” from the engineering question “can the large-model runtime be loaded reliably?”, which makes debugging much easier.